<a href="https://colab.research.google.com/github/tanveeeD/EmergencyCall_Classifier/blob/main/notebooks/04_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers datasets torch scikit-learn

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/EmergencyCallProject/dataset/final_dataset.csv")

df.head()


In [ ]:
label_map = {"fire": 0, "police": 1, "medical": 2}

df["label"] = df["label"].map(label_map)

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["transcript"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42
)

In [ ]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
train_encodings = tokenizer(train_texts, truncation=True, padding=True)
test_encodings = tokenizer(test_texts, truncation=True, padding=True)

In [ ]:
import torch

class EmergencyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = EmergencyDataset(train_encodings, train_labels)
test_dataset = EmergencyDataset(test_encodings, test_labels)

In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_dir="./logs",
    save_strategy="epoch",
    load_best_model_at_end=True
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)

print(classification_report(test_labels, y_pred))

In [ ]:
import torch

text = "Does anyone know CPR?"

device = next(model.parameters()).device

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

inputs = {k: v.to(device) for k, v in inputs.items()}

outputs = model(**inputs)

pred = torch.argmax(outputs.logits, dim=1).item()

reverse_map = {0:"fire", 1:"police", 2:"medical"}

print(reverse_map[pred])

In [ ]:
model.save_pretrained("/content/drive/MyDrive/EmergencyCallProject/models/distilbert_model")
tokenizer.save_pretrained("/content/drive/MyDrive/EmergencyCallProject/models/distilbert_model")